In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
import os
import re
import pandas as pd

sys.path.append(os.path.abspath("../src"))

from settings import INSTANCE_FOLDER
from solvers.FRG import FRGSolver
from solvers.utils import summary

In [ ]:
solver_greedy = FRGSolver()          # greedy puro (beams=0)
solver_bsg    = FRGSolver(beams=10)  # BSG — ajustar beam width aquí

CVS_PATH = INSTANCE_FOLDER / "benchmarks" / "CVS"
H_MAP    = {3: 5, 4: 6, 5: 7, 6: 8, 10: 12}  # H_real → H_solver (H_real + 2)

cvs_folders = sorted(
    [d for d in os.listdir(CVS_PATH) if (CVS_PATH / d).is_dir()]
)

In [ ]:
results = {}
all_g_solved, all_g_steps = [], []
all_b_solved, all_b_steps = [], []

for folder_name in cvs_folders:
    H_real, S_real = [int(x) for x in folder_name.split("-")]
    H           = H_MAP[H_real]
    max_steps   = S_real * H_real * 4
    folder_path = CVS_PATH / folder_name

    # Referencia Excel
    df_ref   = pd.read_excel(folder_path / f"Data{folder_name}.xlsx", header=0)
    ref_dict = dict(zip(df_ref.iloc[:, 0].astype(str), df_ref.iloc[:, 1]))

    # Archivos .dat en orden numérico
    dat_files = sorted(
        [f for f in os.listdir(folder_path) if f.endswith(".dat")],
        key=lambda f: int(re.search(r'-(\d+)\.dat$', f).group(1))
    )

    print(f"\n{'='*72}")
    print(f"  {folder_name}   H={H_real}  S={S_real}  H_solver={H}  max_steps={max_steps}")
    print(f"{'='*72}")
    print(f"{'Instancia':<12} {'Greedy':>8} {'BSG':>8} {'Ref':>8} {'G-Ref':>7} {'B-Ref':>7}")
    print(f"{'-'*55}")

    g_solved, g_steps = [], []
    b_solved, b_steps = [], []

    for filename in dat_files:
        filepath = str(folder_path / filename)
        inst_key = re.sub(r'^data', '', filename.replace('.dat', ''))

        gs, gn = solver_greedy.solve_from_path(filepath, H, max_steps)
        bs, bn = solver_bsg.solve_from_path(filepath, H, max_steps)

        ref_raw = ref_dict.get(inst_key, None)
        ref_val = int(ref_raw) if ref_raw is not None and ref_raw != 0 else None

        g_str  = str(gn) if gs else "NO"
        b_str  = str(bn) if bs else "NO"
        ref_str = str(ref_val) if ref_val else "NO"
        g_diff = f"{gn - ref_val:+d}" if gs and ref_val else "-"
        b_diff = f"{bn - ref_val:+d}" if bs and ref_val else "-"

        print(f"{inst_key:<12} {g_str:>8} {b_str:>8} {ref_str:>8} {g_diff:>7} {b_diff:>7}")

        g_solved.append(gs); g_steps.append(gn)
        b_solved.append(bs); b_steps.append(bn)

    all_g_solved.extend(g_solved); all_g_steps.extend(g_steps)
    all_b_solved.extend(b_solved); all_b_steps.extend(b_steps)

    ref_vals = [int(v) for v in ref_dict.values() if v and v != 0]

    print(f"{'-'*55}")
    print("Greedy:     ", end=""); summary(g_solved, g_steps)
    print("BSG:        ", end=""); summary(b_solved, b_steps)
    if ref_vals:
        print(f"Referencia:  {len(ref_vals)}/40 resueltas  avg={sum(ref_vals)/len(ref_vals):.2f}")

    results[folder_name] = {
        "H": H_real, "S": S_real,
        "g_solved": g_solved, "g_steps": g_steps,
        "b_solved": b_solved, "b_steps": b_steps,
        "ref": ref_dict,
    }

In [ ]:
print("\n" + "="*55)
print("RESUMEN GLOBAL")
print("="*55)
print("Greedy:     ", end=""); summary(all_g_solved, all_g_steps)
print("BSG:        ", end=""); summary(all_b_solved, all_b_steps)

all_ref = [int(v) for r in results.values() for v in r['ref'].values() if v and v != 0]
print(f"Referencia:  {len(all_ref)}/840 resueltas  avg={sum(all_ref)/len(all_ref):.2f}")